[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/Knowledge_Base_Browsing.ipynb)

# Letting an Agent Browse a Knowledge Base

Top-k retrieval answers one question well: *which passages look most like this query?* It cannot
answer *which files contain this exact string?*, *how many are there?*, or *what else is in this
folder?* Those need a different access pattern: browsing.

This notebook builds the second grounding tool the production AI Tutor uses, at notebook scale:

1. Load a real documentation corpus and build a persisted Chroma vector store with Gemini embeddings.
2. Ask a question that top-k retrieval answers incompletely, and watch it fail.
3. Write the same documents out as a **browsable file knowledge base** with three layers.
4. Implement a **read-only sandboxed command runner**: an allowlist, a path jail, a timeout, an
   output cap, and a per-turn command budget.
5. Expose it as **one tool** to Gemini via native function calling and run a small agent loop.
6. Resolve the answer's citations against the evidence a tool actually surfaced, so a remembered
   URL never becomes a trusted source card.

The production sandbox in `app/kb_shell.py` allows eight programs; this one allows six. Everything
else about the shape is the same, and the lesson names each place where we simplified.

## Setup

Pinned versions, current on PyPI as of August 2026.

In [1]:
# Shared install profile for this notebook (pin set checked August 13, 2026)
%pip install -q google-genai==2.18.0 chromadb==1.5.9 huggingface-hub==1.27.0

Note: you may need to restart the kernel to use updated packages.


In [2]:
import getpass
import os

# In Colab, store the key under the key icon in the left sidebar.
if not os.environ.get("GEMINI_API_KEY"):
    try:
        from google.colab import userdata

        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:
        os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")

Two helpers wrap the provider: `generate()` for chat and `embed()` for vectors. Everything below
calls the helpers, so swapping providers means editing this cell and nothing else. Gemini
embeddings default to 3072 dimensions; we ask for 768 and normalize to unit length, which makes
Chroma's default L2 distance rank identically to cosine similarity.

In [3]:
import numpy as np
from google import genai
from google.genai import types

CHAT_MODEL = "gemini-3.5-flash-lite"
EMBED_MODEL = "gemini-embedding-001"

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])


def generate(prompt: str, system: str | None = None) -> str:
    config = types.GenerateContentConfig(system_instruction=system) if system else None
    return client.models.generate_content(model=CHAT_MODEL, contents=prompt, config=config).text


def embed(texts: list[str], task_type: str = "RETRIEVAL_DOCUMENT") -> list[list[float]]:
    vectors: list[list[float]] = []
    for start in range(0, len(texts), 64):  # the API takes batches, not one call per text
        response = client.models.embed_content(
            model=EMBED_MODEL,
            contents=texts[start : start + 64],
            config=types.EmbedContentConfig(task_type=task_type, output_dimensionality=768),
        )
        for item in response.embeddings:
            values = np.asarray(item.values, dtype=np.float32)
            vectors.append((values / np.linalg.norm(values)).tolist())
    return vectors


print(generate("Reply with exactly: setup ok"))

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


setup ok


## 1. Load the corpus

We reuse the course knowledge dataset: one JSONL file, one row per documentation page, with
`name`, `url`, `source`, and `content`. Plain Python, no framework.

We slice it to two sources, HuggingFace `peft` and `trl`, so the whole notebook stays small enough
to run in a couple of minutes.

In [4]:
import json

from huggingface_hub import hf_hub_download

corpus_path = hf_hub_download(
    repo_id="jaiganesan/ai_tutor_knowledge",
    filename="ai_tutor_knowledge.jsonl",
    repo_type="dataset",
)
corpus = [json.loads(line) for line in open(corpus_path, encoding="utf-8") if line.strip()]

KB_SOURCES = ("peft", "trl")
docs = [d for d in corpus if d["source"] in KB_SOURCES]

print(f"corpus: {len(corpus)} pages | our slice: {len(docs)} pages")
print("fields:", list(docs[0]))
print("example:", docs[0]["name"], "|", docs[0]["url"])

corpus: 762 pages | our slice: 82 pages
fields: ['tokens', 'doc_id', 'name', 'url', 'source', 'content']
example: Command Line Interfaces (CLIs) | https://huggingface.co/docs/trl/clis


## 2. Chunk, embed, and persist a Chroma store

Fixed-size chunking with overlap, the same shape you built in Part 1. `PersistentClient` writes the
store to disk, so the embedding cost is paid once and later runs load it instead of re-embedding.

In [5]:
import shutil
from pathlib import Path

import chromadb


def split(text: str, size: int = 1200, overlap: int = 150) -> list[str]:
    pieces, start = [], 0
    while start < len(text):
        pieces.append(text[start : start + size])
        start += size - overlap
    return pieces


chunks, metadatas = [], []
for doc in docs:
    for part, piece in enumerate(split(doc["content"])):
        chunks.append(piece)
        metadatas.append({"title": doc["name"], "source": doc["source"], "part": part})

print(f"{len(docs)} pages -> {len(chunks)} chunks")

82 pages -> 529 chunks


In [6]:
STORE_DIR = Path("kb_demo_chroma")
shutil.rmtree(STORE_DIR, ignore_errors=True)  # rebuild from scratch so re-runs are deterministic
collection = chromadb.PersistentClient(path=str(STORE_DIR)).create_collection("ai_tutor_kb")

vectors = embed(chunks)
collection.add(
    ids=[f"c{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=vectors,
    metadatas=metadatas,
)
print(f"{collection.count()} rows in the store, {len(vectors[0])} dimensions each")

529 rows in the store, 768 dimensions each


## 3. Where top-k runs out

Here is a question a student actually asks: *which TRL trainers can I wire LoRA into?* In the
corpus that reduces to an exact-string question, which pages pass a `peft_config` argument, and it
demands an **exhaustive** answer.

Top-k returns the four chunks that look most like the query. Those chunks are relevant. They are
also not the whole answer, and nothing in the payload tells the model that.

In [7]:
QUESTION = (
    "I want to fine-tune with LoRA using TRL. Which pages of the TRL documentation "
    "actually pass a `peft_config` argument? List every one of them."
)


def top_k(question: str, k: int = 4):
    query_vector = embed([question], task_type="RETRIEVAL_QUERY")[0]
    hits = collection.query(query_embeddings=[query_vector], n_results=k)
    return list(zip(hits["documents"][0], hits["metadatas"][0]))


hits = top_k(QUESTION)
for _text, meta in hits:
    print(f"[{meta['source']}] {meta['title'][:60]} (part {meta['part']})")

[trl] Examples of using peft with trl to finetune 8-bit models wit (part 0)
[trl] Examples of using peft with trl to finetune 8-bit models wit (part 1)
[trl] Command Line Interfaces (CLIs) (part 1)
[trl] Examples of using peft with trl to finetune 8-bit models wit (part 2)


In [8]:
context = "\n\n---\n\n".join(f"# {m['title']}\n{t}" for t, m in hits)
print(generate(f"Context:\n{context}\n\nQuestion: {QUESTION}\n\nAnswer using ONLY the context above."))

Based on the provided context, the pages/sections that pass a `peft_config` argument are:

1. **Examples of using peft with trl to finetune 8-bit models with Low Rank Adaption (LoRA)** (specifically in the code examples showing how to initialize `AutoModelForCausalLMWithValueHead.from_pretrained` with `peft_config=lora_config` and when loading in 8-bit or 4-bit precision).


Three of the four chunks came from a single page, so the model saw one page and answered with one
page. Hold on to that answer, we will check it against ground truth at the end.

The failure is structural, not a tuning problem. Raising `k` pulls in more near-duplicates before
it pulls in the missing pages, because ranking by similarity is the wrong operation for a question
whose answer is a **set of files**.

## 4. Build the browsable file knowledge base

The production KB has three layers, and we mirror all three:

- **`raw/`** holds a markdown mirror of every corpus page, with the title, source, and canonical URL
  in frontmatter. Read-only, and the local source of authority.
- **`generated/`** holds machine indexes. Production ships `corpus_manifest.jsonl`,
  `headings.jsonl`, and `symbols.tsv`; we build the manifest, one JSON row per page.
- **`wiki/`** holds synthesis and navigation. Production has 29 LLM-maintained pages; we scaffold a
  single `index.md` so the agent has a map to open first.

In [9]:
import re

KB = Path("kb_demo")


def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")[:60] or "untitled"


def build_file_kb(docs: list[dict], root: Path = KB) -> list[dict]:
    shutil.rmtree(root, ignore_errors=True)
    (root / "generated").mkdir(parents=True)
    (root / "wiki").mkdir(parents=True)

    manifest, taken = [], set()
    for doc in docs:
        name, suffix = slugify(doc["name"]), 1
        while (doc["source"], name) in taken:  # two pages can slugify to the same name
            suffix += 1
            name = f"{slugify(doc['name'])}-{suffix}"
        taken.add((doc["source"], name))

        relative = f"raw/docs/{doc['source']}/{name}.md"
        page = root / relative
        page.parent.mkdir(parents=True, exist_ok=True)
        page.write_text(
            f"---\ntitle: {doc['name']}\nsource: {doc['source']}\nurl: {doc['url']}\n---\n\n"
            f"{doc['content']}\n",
            encoding="utf-8",
        )
        manifest.append(
            {"title": doc["name"], "source": doc["source"], "path": relative, "url": doc["url"]}
        )

    (root / "generated" / "corpus_manifest.jsonl").write_text(
        "".join(json.dumps(row) + "\n" for row in manifest), encoding="utf-8"
    )

    counts: dict[str, int] = {}
    for row in manifest:
        counts[row["source"]] = counts.get(row["source"], 0) + 1
    index = ["# KB Index", "", "Navigation layer over the raw page mirrors.", "", "## Sources", ""]
    index += [f"- **{s}** - `raw/docs/{s}/` ({c} pages)" for s, c in sorted(counts.items())]
    index += [
        "",
        "## Generated indexes",
        "",
        "- `generated/corpus_manifest.jsonl` - one JSON row per page (title, source, path, url).",
        "",
    ]
    (root / "wiki" / "index.md").write_text("\n".join(index), encoding="utf-8")
    return manifest


manifest = build_file_kb(docs)
print(f"wrote {len(manifest)} pages")
print((KB / "wiki" / "index.md").read_text())

wrote 82 pages
# KB Index

Navigation layer over the raw page mirrors.

## Sources

- **peft** - `raw/docs/peft/` (48 pages)
- **trl** - `raw/docs/trl/` (34 pages)

## Generated indexes

- `generated/corpus_manifest.jsonl` - one JSON row per page (title, source, path, url).



## 5. The sandbox

The tool must let a language model run commands without letting it run *anything*. Five constraints
carry that, and each one maps to a constant in `app/kb_shell.py`:

| Constraint | Here | Production |
|---|---|---|
| Allowed programs | 6: `ls cat grep find head wc` | 8: `rg grep find ls sed head cat wc` |
| Allowed flags | per-program allowlist | per-program allowlist |
| Shell syntax | no pipes, redirects, chaining, expansion | same |
| Path jail | must resolve inside the KB root | same |
| Timeout | 5s | 8s (capped at 30s) |
| Output cap | 4,000 chars | 40,000 chars (capped at 80,000) |
| Command budget | 10 per question | 20 per turn |

Note what the allowlist implies: there is no write command, no network command, and no interpreter.
Read-only is a property of the list, not of a check we run afterwards.

In [10]:
import shlex
import subprocess

ALLOWED_FLAGS = {
    "ls": {"-1", "-a", "-l"},
    "cat": set(),
    "grep": {"-r", "-n", "-i", "-l", "-c", "-w", "-F", "-m"},
    "find": {"-maxdepth", "-mindepth", "-type", "-name", "-iname"},
    "head": {"-n"},
    "wc": {"-l", "-w", "-c"},
}
VALUE_FLAGS = {"-m", "-n", "-maxdepth", "-mindepth", "-type", "-name", "-iname"}
SHELL_TOKENS = {"|", "||", "&", "&&", ";", ">", ">>", "<", "`"}

TIMEOUT_SECONDS = 5
MAX_OUTPUT_CHARS = 4_000
COMMAND_BUDGET = 10


class KbCommandError(ValueError):
    """The command violated the read-only policy."""


def jail_path(value: str, root: Path) -> str:
    resolved = Path(value).resolve() if Path(value).is_absolute() else (root / value).resolve()
    if resolved != root and root not in resolved.parents:
        raise KbCommandError(f"path escapes the KB root: {value}")
    if not resolved.exists():
        raise KbCommandError(
            f"path does not exist: {value} "
            "(shell globs are not expanded; pass a directory and let grep recurse)"
        )
    return "." if resolved == root else str(resolved.relative_to(root))

`build_argv` turns a command string into an argument vector, or raises. It never builds a shell
string, so there is nothing for a metacharacter to escape into. The one subtlety is positional
parsing: `grep`'s first positional is a search **pattern**, not a path, so jailing every positional
would reject `grep -r peft_config raw/docs/trl` as a missing file.

In [11]:
def build_argv(command: str, root: Path) -> list[str]:
    if "\n" in command or "$(" in command or "${" in command:
        raise KbCommandError("one command per call, and no shell expansion")
    tokens = shlex.split(command)
    if not tokens:
        raise KbCommandError("empty command")
    if any(token in SHELL_TOKENS for token in tokens):
        raise KbCommandError("no pipes, redirects, or command chaining")

    program, arguments = tokens[0], tokens[1:]
    if program not in ALLOWED_FLAGS:
        raise KbCommandError(
            f"unsupported command `{program}`. allowed: {', '.join(sorted(ALLOWED_FLAGS))}"
        )

    argv, positional, expecting_value = [program], 0, False
    for token in arguments:
        if expecting_value:  # e.g. the N in `-m N`, never a path
            argv.append(token)
            expecting_value = False
        elif token.startswith("-"):
            if token not in ALLOWED_FLAGS[program] and not re.fullmatch(r"-\d+", token):
                raise KbCommandError(
                    f"unsupported {program} flag `{token}`. "
                    f"allowed: {' '.join(sorted(ALLOWED_FLAGS[program])) or 'none'}"
                )
            argv.append(token)
            expecting_value = token in VALUE_FLAGS
        else:
            positional += 1
            is_path = not (program == "grep" and positional == 1)
            argv.append(jail_path(token, root) if is_path else token)

    if program == "grep":
        if positional < 2:
            argv.append(".")
        if "-r" not in argv:
            argv.insert(1, "-r")
    return argv

`KbShell` owns the per-question state: the jail root and the command budget. One instance per
question, so an agent that browses forever hits a wall instead of running until the API times out.

In [12]:
class KbShell:
    def __init__(self, root: Path, budget: int = COMMAND_BUDGET):
        self.root = Path(root).resolve()
        self.budget = budget
        self.used = 0
        self.commands: list[str] = []

    def run(self, command: str) -> str:
        if self.used >= self.budget:
            return (
                f"$ {command}\nerror: command budget exhausted "
                f"({self.used}/{self.budget}). Answer with what you already have."
            )
        self.used += 1
        self.commands.append(command)

        try:
            argv = build_argv(command, self.root)
        except (KbCommandError, ValueError) as error:
            return f"$ {command}\nerror: {error}"  # the error text teaches the model the form
        try:
            done = subprocess.run(
                argv,
                cwd=self.root,
                capture_output=True,
                text=True,
                timeout=TIMEOUT_SECONDS,
                # A scrubbed environment: the child inherits no secrets from the
                # notebook process, and no colour settings that would put ANSI
                # escapes into the model's context.
                env={"HOME": os.environ.get("HOME", ""), "PATH": os.environ.get("PATH", ""),
                     "LC_ALL": "C.UTF-8"},
            )
        except subprocess.TimeoutExpired:
            return f"$ {command}\nerror: timed out after {TIMEOUT_SECONDS}s"

        output = done.stdout + (f"\n[stderr] {done.stderr}" if done.stderr else "")
        if len(output) > MAX_OUTPUT_CHARS:
            output = output[:MAX_OUTPUT_CHARS] + f"\n... truncated at {MAX_OUTPUT_CHARS} chars ..."
        return f"$ {command}\nexit_code: {done.returncode}\n{output.strip()}"

Let's confirm the policy holds. The first three commands are the ones the agent needs; the last
four are the attacks the sandbox exists for.

In [13]:
smoke = KbShell(KB, budget=20)
for command in [
    "ls raw/docs",
    "wc -l generated/corpus_manifest.jsonl",
    "grep -r -l peft_config raw/docs/trl",
    "cat ../../../etc/passwd",
    "cat wiki/index.md > /tmp/leak.md",
    "curl https://example.com",
    "grep -r --perl-regexp peft raw/docs",
]:
    print(smoke.run(command)[:400])
    print()

$ ls raw/docs
exit_code: 0
peft
trl

$ wc -l generated/corpus_manifest.jsonl
exit_code: 0
82 generated/corpus_manifest.jsonl

$ grep -r -l peft_config raw/docs/trl
exit_code: 0
raw/docs/trl/using-llama-models-with-trl.md
raw/docs/trl/reward-modeling.md
raw/docs/trl/multi-adapter-rl-marl-a-single-base-model-for-everything.md
raw/docs/trl/supervised-fine-tuning-trainer.md
raw/docs/trl/examples-of-using-peft-with-trl-to-finetune-8-bit-models-wit.md

$ cat ../../../etc/passwd
error: path escapes the KB root: ../../../etc/passwd

$ cat wiki/index.md > /tmp/leak.md
error: no pipes, redirects, or command chaining

$ curl https://example.com
error: unsupported command `curl`. allowed: cat, find, grep, head, ls, wc

$ grep -r --perl-regexp peft raw/docs
error: unsupported grep flag `--perl-regexp`. allowed: -F -c -i -l -m -n -r -w



Every rejection names the supported form. That matters more than it looks: the error text is the
only documentation the model gets at call time, so a vague `invalid command` costs a retry from the
budget while a specific one gets the next command right.

## 6. One tool

The whole sandbox is exposed as a single function with a single string parameter. The description
is the model's entire manual, so it carries the allowed programs, the prohibitions, and the KB
layout.

In [14]:
KB_TOOL = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="run_kb_command",
            description=(
                "Run ONE read-only command inside the local knowledge base. Allowed "
                "programs: ls, cat, grep, find, head, wc. No pipes, redirects, chaining, "
                "or shell globs. Paths are relative to the KB root and cannot escape it. "
                "Layout: wiki/index.md (map), generated/corpus_manifest.jsonl (one row "
                "per page), raw/docs/<source>/<page>.md (the page mirrors)."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "command": types.Schema(
                        type=types.Type.STRING,
                        description="e.g. `grep -r -l peft_config raw/docs/trl`",
                    )
                },
                required=["command"],
            ),
        )
    ]
)

### The same tool on OpenAI and Anthropic

Provided for reference and **not executed** in this notebook, which runs the Gemini path only. The
sandbox and the loop are unchanged; only the declaration and the result envelope differ.

```python
# OpenAI, Responses API (model: gpt-5.6-luna)
kb_tool_openai = {
    "type": "function",
    "name": "run_kb_command",
    "description": "Run ONE read-only command inside the local knowledge base. ...",
    "parameters": {
        "type": "object",
        "properties": {"command": {"type": "string"}},
        "required": ["command"],
    },
}
# A call arrives as an output item of type "function_call" with .call_id, .name, .arguments
# (a JSON string). You append {"type": "function_call_output", "call_id": ..., "output": ...}.

# Anthropic, Messages API (model: claude-opus-5)
kb_tool_anthropic = {
    "name": "run_kb_command",
    "description": "Run ONE read-only command inside the local knowledge base. ...",
    "input_schema": {
        "type": "object",
        "properties": {"command": {"type": "string"}},
        "required": ["command"],
    },
}
# Loop while response.stop_reason == "tool_use"; each tool_use block carries .id, .name, .input
# (already parsed). You reply with a user message of {"type": "tool_result", "tool_use_id": ...}.
```

All three are the same three things: a name, a description the model reads as documentation, and a
JSON Schema for the arguments.

## 7. The agent loop

Same loop you built in Part 2: call the model, execute any tool calls it returns, append the
results, repeat until it answers with text instead of a call. Two additions matter for browsing.

First, the **first-command rule**: open the wiki index before searching. Production enforces this
in `data/kb/AGENTS.md` because unguided `grep` over the raw tree burns budget on noise.

Second, an explicit **stop condition**. A model with a shell will keep browsing past the point where
it has the answer, and every extra round costs a full context re-send.

In [15]:
SYSTEM = """You are a tutor grounded in a local knowledge base you browse with run_kb_command.

- Start with `cat wiki/index.md` to learn the layout.
- Use `grep -r -l PATTERN raw/docs/<source>` to list the files containing an exact string.
- Shell globs are not expanded. Pass a directory and let grep recurse.
- Stop browsing as soon as a command has answered the question. Never repeat a command.
- You get at most 10 commands. Cite the raw/ path of every page you used."""


def browse_and_answer(question: str, max_steps: int = 8, verbose: bool = True):
    shell = KbShell(KB)
    history = [types.Content(role="user", parts=[types.Part(text=question)])]
    config = types.GenerateContentConfig(system_instruction=SYSTEM, tools=[KB_TOOL])

    for step in range(1, max_steps + 1):
        reply = client.models.generate_content(
            model=CHAT_MODEL, contents=history, config=config
        ).candidates[0]
        history.append(reply.content)

        calls = [p.function_call for p in (reply.content.parts or []) if p.function_call]
        if not calls:  # no tool call means the model is answering
            return "".join(p.text for p in (reply.content.parts or []) if p.text), shell

        results = []
        for call in calls:
            output = shell.run(call.args.get("command", ""))
            if verbose:
                lines = output.splitlines()
                print(f"step {step}: {lines[0]}")
                for line in lines[2:5]:
                    print(f"         {line[:96]}")
            results.append(
                types.Part.from_function_response(name=call.name, response={"output": output})
            )
        history.append(types.Content(role="user", parts=results))

    return "(step limit reached)", shell

In [16]:
answer, shell = browse_and_answer(QUESTION)
print(f"\ncommands used: {shell.used}/{shell.budget}\n")
print(answer)

step 1: $ cat wiki/index.md
         # KB Index
         
         Navigation layer over the raw page mirrors.


step 2: $ grep -r -l peft_config raw/docs/trl
         raw/docs/trl/using-llama-models-with-trl.md
         raw/docs/trl/reward-modeling.md
         raw/docs/trl/multi-adapter-rl-marl-a-single-base-model-for-everything.md



commands used: 2/10

Here are the pages of the TRL documentation that pass a `peft_config` argument:

1. `raw/docs/trl/using-llama-models-with-trl.md`
2. `raw/docs/trl/reward-modeling.md`
3. `raw/docs/trl/multi-adapter-rl-marl-a-single-base-model-for-everything.md`
4. `raw/docs/trl/supervised-fine-tuning-trainer.md`
5. `raw/docs/trl/examples-of-using-peft-with-trl-to-finetune-8-bit-models-wit.md`


Now check the answer against ground truth we compute directly from disk, the same way a retrieval
eval scores `recall@k`.

In [17]:
truth = sorted(
    row["path"]
    for row in manifest
    if row["source"] == "trl" and "peft_config" in (KB / row["path"]).read_text(encoding="utf-8")
)
cited = sorted(set(re.findall(r"raw/docs/[\w./-]+\.md", answer)))

print("ground truth:")
for path in truth:
    print("  ", path)
print(f"\ncited: {len(cited)} | correct: {len(set(cited) & set(truth))} | wrong: {sorted(set(cited) - set(truth))}")

ground truth:
   raw/docs/trl/examples-of-using-peft-with-trl-to-finetune-8-bit-models-wit.md
   raw/docs/trl/multi-adapter-rl-marl-a-single-base-model-for-everything.md
   raw/docs/trl/reward-modeling.md
   raw/docs/trl/supervised-fine-tuning-trainer.md
   raw/docs/trl/using-llama-models-with-trl.md

cited: 5 | correct: 5 | wrong: []


Two commands, complete answer. The retrieval arm got one page out of five from four chunks, and
neither arm was wrong about what it saw: they were answering different questions. That is the whole
argument for a second grounding tool.

The cost is visible in the trace. Browsing spent two model calls where retrieval spent one, and in
production that difference is larger, roughly 7.7 KB commands against 0.9 retrieval calls per turn.
Latency is the price of exhaustiveness.

## 8. Citation resolution

The agent cited five paths and all five were real. Nothing so far guarantees that. A model can
write a plausible documentation URL from memory, and it will look exactly like a citation it earned.

So we resolve citations against evidence instead of trusting them. A reference becomes a **trusted
source card** only if two things hold: it resolves to a real manifest entry, and a tool actually
surfaced it during this question. Everything else stays a plain link in the prose.

In [18]:
BY_PATH = {row["path"]: row for row in manifest}
BY_URL = {row["url"]: row for row in manifest}

RAW_PATH_RE = re.compile(r"raw/docs/[\w./-]+\.md")
LINK_RE = re.compile(r"\[([^\]]+)\]\(([^)\s]+)\)")
BARE_URL_RE = re.compile(r"https?://[^\s<>()`\]]+")


def parse_citations(answer: str) -> list[str]:
    refs = [m.group(2) for m in LINK_RE.finditer(answer)]
    linked = set(refs)
    refs += [u.rstrip(".,;:") for u in BARE_URL_RE.findall(answer) if u.rstrip(".,;:") not in linked]
    refs += [p for p in RAW_PATH_RE.findall(answer) if p not in linked]
    seen, ordered = set(), []
    for ref in refs:
        if ref not in seen:
            seen.add(ref)
            ordered.append(ref)
    return ordered


def shell_evidence(shell: KbShell, outputs: list[str]) -> set[str]:
    """Pages a KB command read as an argument or returned as a search hit."""
    touched = set()
    for command, output in zip(shell.commands, outputs):
        for path in RAW_PATH_RE.findall(command) + RAW_PATH_RE.findall(output):
            if path in BY_PATH:
                touched.add(path)
    return touched


def resolve_citations(answer: str, evidence: set[str]):
    cards, plain = [], []
    for ref in parse_citations(answer):
        row = BY_PATH.get(ref) or BY_URL.get(ref)
        if row and row["path"] in evidence:
            cards.append(row)
        else:
            plain.append(ref)  # a plausible reference is still not an earned one
    return cards, plain

To make the difference visible, we test on an answer that mixes both kinds of citation: two pages
the shell really returned, one real-looking HuggingFace URL the model could have produced from
memory, and one real KB path that this question never opened.

In [19]:
commands = ["cat wiki/index.md", "grep -r -l peft_config raw/docs/trl"]
outputs = [
    "# KB Index ...",
    "raw/docs/trl/reward-modeling.md\nraw/docs/trl/supervised-fine-tuning-trainer.md",
]
replayed = KbShell(KB)
replayed.commands = commands
evidence = shell_evidence(replayed, outputs)

draft = """TRL wires LoRA through `peft_config`. The reward trainer accepts it
(raw/docs/trl/reward-modeling.md), and so does the SFT trainer
(raw/docs/trl/supervised-fine-tuning-trainer.md). The DPO trainer supports it too, see
[DPO Trainer](https://huggingface.co/docs/trl/dpo_trainer), and PEFT's own
[LoRA guide](raw/docs/peft/lora.md) covers the config object."""

cards, plain = resolve_citations(draft, evidence)
print("trusted source cards:")
for row in cards:
    print("  +", row["path"], "|", row["title"][:50])
print("\nleft as plain links (no tool surfaced them this turn):")
for ref in plain:
    print("  -", ref)

trusted source cards:
  + raw/docs/trl/reward-modeling.md | Reward Modeling
  + raw/docs/trl/supervised-fine-tuning-trainer.md | Supervised Fine-tuning Trainer

left as plain links (no tool surfaced them this turn):
  - https://huggingface.co/docs/trl/dpo_trainer
  - raw/docs/peft/lora.md


Both rejected references are *plausible*. `https://huggingface.co/docs/trl/dpo_trainer` is a real
page, and `raw/docs/peft/lora.md` is a real file in this KB. Neither was opened while answering this
question, so neither earns a card. The rule is about provenance, not plausibility, which is why it
catches hallucinated citations that a URL-validity check would wave through.

Production applies the same rule across three evidence buckets in `resolve_answer_citations`
(`app/chat_service.py`): retrieval results, browsed KB files, and web-tool results.

## What you built, and what it leaves out

Two grounding tools over one corpus, plus the machinery that keeps the second one safe:

- A persisted Chroma store with Gemini embeddings, and a question it answers incompletely.
- A three-layer file KB: raw mirrors, a generated manifest, a wiki index.
- A read-only command runner with an allowlist, a path jail, a timeout, an output cap, and a budget.
- One tool, native function calling, and a browsing loop that answered exhaustively in two commands.
- Citation resolution that turns only tool-surfaced references into trusted cards.

What the production system adds: `rg` and `sed`, per-flag parsing for eight programs, capped reader
threads so a large `cat` cannot spike memory, `--` before every search pattern so a pattern
beginning with `-` can never be re-read as a flag, and a rule that broad searches over `raw/` must
carry `-m N`. Read `app/kb_shell.py`; it is about 530 lines and every one of them is a hostile input
someone thought of.

### Exercises

1. **Add the wiki layer.** Write one `wiki/topics/lora.md` summarizing the LoRA material, with a
   "where to look first" list of raw paths. Re-run the agent and count commands used.
2. **Add `sed -n START,ENDp`.** Accept only that exact form, so the model can read a line range
   without `cat`-ing a 30,000-character page into context.
3. **Break the jail.** Try `grep -r peft /etc`, a symlink out of `raw/`, and `cat "$(pwd)/x"`. Each
   should be rejected before `subprocess.run`, not after.
4. **Measure the trade-off.** Run five questions through both arms and record commands, model calls,
   and wall-clock time. That table is the honest version of "browsing is better".